# Multimodal emotion recognition on TESS (speech, text, fusion)

1. **Add data:** the dataset *Toronto emotional speech set (TESS)* (`ejlok1/toronto-emotional-speech-set-tess`).
2. **Settings:** Accelerator *GPU T4 x1* (or P100), Internet *on* (downloads WavLM and BERT).
3. **Save Version > Save & Run All (Commit).** About 1.5-2 hours.
4. Download `TESS_results.zip` from the Output tab.

In [1]:
CODE_B64 = '...'
CODE_BUILD = '2026-11-16 21:37 2391b93e'

In [2]:
import base64, io, os, sys, tarfile, subprocess, time, torch
PROJ = '/kaggle/working/project'
os.makedirs(PROJ, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode='r:gz').extractall(PROJ)
print('code build:', CODE_BUILD)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (enable a GPU!)')
import transformers, librosa, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| librosa',
      librosa.__version__, '| sklearn', sklearn.__version__)
wavs = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f.lower().endswith('.wav')]
print(len(wavs), 'wav files under /kaggle/input, e.g.', wavs[:2])
assert wavs, 'TESS not found: add the dataset (Add data -> ejlok1/toronto-emotional-speech-set-tess)'

/tmp/ipykernel_50/1871701724.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode='r:gz').extractall(PROJ)


code build: 2026-11-16 21:37 2391b93e
GPU: Tesla T4
torch 2.11.0+cu130 | transformers 5.20.2 | librosa 0.11.0 | sklearn 1.6.1
5600 wav files under /kaggle/input, e.g. ['/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess/TESS Toronto emotional speech set data/OAF_Fear/OAF_back_fear.wav', '/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess/TESS Toronto emotional speech set data/OAF_Fear/OAF_bar_fear.wav']


In [3]:
def run(cmd):
    print('+', cmd, flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, cwd=PROJ)
    print(f'({time.time() - t0:.0f} s)')
    assert p.returncode == 0, f'failed: {cmd}'

FEAT = '/tmp/tess_features'   # about 7 GB, not part of the output
run(f'python scripts/extract_features.py --tess /kaggle/input --out {FEAT}')

+ python scripts/extract_features.py --tess /kaggle/input --out /tmp/tess_features


skipped 2 files with unexpected names, e.g. ['OA_bite_neutral.wav', 'OA_bite_neutral.wav']
2799 clips, 200 words, speakers ['OAF', 'YAF']
clips per speaker x emotion: {('OAF', 'angry'): 200, ('OAF', 'disgust'): 200, ('OAF', 'fear'): 200, ('OAF', 'happy'): 200, ('OAF', 'neutral'): 199, ('OAF', 'ps'): 200, ('OAF', 'sad'): 200, ('YAF', 'angry'): 200, ('YAF', 'disgust'): 200, ('YAF', 'fear'): 200, ('YAF', 'happy'): 200, ('YAF', 'neutral'): 200, ('YAF', 'ps'): 200, ('YAF', 'sad'): 200}
1 words do not have exactly 14 clips (2 speakers x 7 emotions), e.g. {'bite': 13}
split word: train 1959, val 420, test 420
split oaf2yaf: train 1189, val 210, test 1400
split yaf2oaf: train 1190, val 210, test 1399


Loading weights: 100%|██████████| 248/248 [00:00<00:00, 10890.30it/s]


speech 500/2799 (68 s)
speech 1000/2799 (98 s)
speech 1500/2799 (126 s)
speech 2000/2799 (154 s)
speech 2500/2799 (182 s)
trimmed clip length: mean 1.97 s, min 1.06, max 2.98
noisy test set at 20 dB: 420 clips
noisy test set at 10 dB: 420 clips
noisy test set at 0 dB: 420 clips


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2671.22it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


text example: 'Say the word back.' -> ['say', 'the', 'word', 'back']
features done in 278 s -> /tmp/tess_features
(288 s)


In [4]:
run(f'python scripts/run_all.py --features {FEAT} --results Results --seeds 0 1 2 --epochs 40')

+ python scripts/run_all.py --features /tmp/tess_features --results Results --seeds 0 1 2 --epochs 40
speech (wavlm) word seed 0: accuracy 0.9952 macro-F1 0.9952 | noise {'0': {'accuracy': 0.4785714285714286, 'macro_f1': 0.35807621599740524}, '10': {'accuracy': 0.8, 'macro_f1': 0.7641349815017792}, '20': {'accuracy': 0.9166666666666666, 'macro_f1': 0.9118015673279345}}
speech (wavlm) word seed 1: accuracy 1.0000 macro-F1 1.0000 | noise {'0': {'accuracy': 0.5095238095238095, 'macro_f1': 0.4932283042516556}, '10': {'accuracy': 0.9166666666666666, 'macro_f1': 0.9109861949214276}, '20': {'accuracy': 0.9928571428571429, 'macro_f1': 0.9928526757795051}}
speech (wavlm) word seed 2: accuracy 1.0000 macro-F1 1.0000 | noise {'0': {'accuracy': 0.5214285714285715, 'macro_f1': 0.4794122597220576}, '10': {'accuracy': 0.9119047619047619, 'macro_f1': 0.9047187208988515}, '20': {'accuracy': 0.9904761904761905, 'macro_f1': 0.9904333206816645}}
speech (wavlm) oaf2yaf seed 0: accuracy 0.6964 macro-F1 0.67

In [5]:
run(f'python scripts/text_zero_shot.py --features {FEAT} --results Results')

+ python scripts/text_zero_shot.py --features /tmp/tess_features --results Results


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7554.06it/s]


{
 "clips": 2799,
 "sentences": 200,
 "upper_bound_any_text_model": 0.1429081814933905,
 "zero_shot_model": "j-hartmann/emotion-english-distilroberta-base",
 "zero_shot_accuracy": 0.1429081814933905,
 "zero_shot_predicted_label_share": {
  "angry": 0.04,
  "disgust": 0.055,
  "fear": 0.015,
  "happy": 0.0,
  "neutral": 0.875,
  "ps": 0.0,
  "sad": 0.015
 }
}
(15 s)


In [6]:
run('python scripts/make_results.py --results Results')
print(open(f'{PROJ}/Results/accuracy_table.md').read())

+ python scripts/make_results.py --results Results
| Variant | unseen words: accuracy | macro-F1 | OAF -> YAF: accuracy | macro-F1 | YAF -> OAF: accuracy | macro-F1 |
|---|---|---|---|---|---|---|
| speech | 99.8 ± 0.2 | 99.8 ± 0.2 | 65.4 ± 7.3 | 62.3 ± 7.4 | 60.2 ± 1.9 | 52.4 ± 3.0 |
| speech_logmel | 99.4 ± 0.3 | 99.4 ± 0.3 | 32.2 ± 4.3 | 28.0 ± 4.8 | 35.9 ± 1.4 | 26.3 ± 1.9 |
| speech_spknorm | 99.9 ± 0.1 | 99.9 ± 0.1 | 56.4 ± 4.2 | 53.7 ± 3.8 | 51.3 ± 1.5 | 45.6 ± 4.6 |
| text | 14.3 ± 0.0 | 3.6 ± 0.0 | 14.3 ± 0.0 | 3.6 ± 0.0 | 14.3 ± 0.0 | 3.6 ± 0.0 |
| fusion | 99.1 ± 0.1 | 99.1 ± 0.1 | 59.5 ± 11.2 | 56.7 ± 13.8 | 57.0 ± 2.7 | 50.8 ± 2.9 |
| fusion_concat | 99.8 ± 0.1 | 99.8 ± 0.1 | 57.6 ± 11.1 | 54.1 ± 11.4 | 45.9 ± 8.1 | 37.4 ± 8.5 |
| fusion_xattn | 99.5 ± 0.2 | 99.5 ± 0.2 | 52.1 ± 12.4 | 50.0 ± 12.4 | 52.1 ± 1.7 | 44.7 ± 2.2 |

Mean ± standard deviation over 3 seeds, in %. Chance level is 14.3 % (7 balanced classes).
tables and plots -> Results
(39 s)
| Variant | unseen words

In [7]:
# keep weights of seed 0 for the three main models; every run keeps metrics, predictions, reps, logs
import glob, zipfile
keep = {f'Results/runs/{v}/{s}/seed0/model.pt'
        for v in ('speech', 'speech_spknorm', 'text', 'fusion', 'fusion_xattn')
        for s in ('word', 'oaf2yaf', 'yaf2oaf')}
out = '/kaggle/working/TESS_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{PROJ}/Results/**/*', recursive=True):
        rel = os.path.relpath(p, PROJ).replace(os.sep, '/')
        if os.path.isfile(p) and (not rel.endswith('model.pt') or rel in keep):
            z.write(p, rel)
    for f in ('metadata.json', 'splits.json', 'text_tokens.json'):
        z.write(f'{FEAT}/{f}', f'Results/data/{f}')
print(out, round(os.path.getsize(out) / 1e6, 1), 'MB | code build', CODE_BUILD)

/kaggle/working/TESS_results.zip 163.7 MB | code build 2026-11-16 21:37 2391b93e
